# 03. Offer Funnel

## Purpose
Follow every offer a customer received and record whether it was **viewed** and whether it was **completed after being viewed**. This table is the basis for offer performance analysis in SQL, Python and Power BI.

## Inputs
- `events_clean.csv`, `offers_clean.csv` from `01_data_cleaning`

## Output
- `offer_funnel.csv`: one row per offer received (66,501 rows)

## Key definitions
| Term | Meaning |
|---|---|
| Window | from the time the offer was received to `received_time + duration x 24` hours |
| Viewed | the customer viewed the offer inside its window |
| Completed (any) | the customer completed the offer inside its window, whether or not they viewed it |
| Influenced completion | completed **at or after** the first view. The customer knew about the offer before completing it |

## Method decisions
1. **Time in hours.** Events are recorded in whole hours, so we compare hours directly instead of the decimal `time_days` column.
2. **Overlapping receipts.** The same customer can receive the same offer more than once, and the windows can overlap. A plain join would count one view or completion toward several receipts. We use `merge_asof` so each event is matched to the **most recent** receipt before it, and is counted once.
3. **Same-hour ties.** If a view and a completion fall in the same hour, the order is unknown. We count it as influenced (`>=`), because customers often open an offer and buy in the same visit.

## Limitations
- "Influenced" describes **sequence**, not proof of cause. The customer may have bought anyway.
- Informational offers have no `offer completed` events, so this funnel cannot measure their effect. That is handled in `04_informational_offer`.

## 1. Load the cleaned datasets

In [25]:
import pandas as pd
import numpy as np

# cleaned files created in 01_data_cleaning
customers = pd.read_csv('../data/cleaned/customers_clean.csv')   # not used in this notebook
offers = pd.read_csv('../data/cleaned/offers_clean.csv')
events = pd.read_csv('../data/cleaned/events_clean.csv')

## 2. Build the offer funnel

**Goal:** for every `offer received` event, work out how far the offer got:
- was it **viewed** within its duration window?
- was it **completed after being viewed**? (a truly "influenced" completion, not just a coincidental purchase that happened to clear the spending threshold)

### Step 2.1: one row per received offer
Each received offer gets a time window: from the moment it is received until it expires.

`window_end = received_time + duration x 24` (duration is in days, time is in hours).

Example: an offer with a 3-day duration received at hour 0 expires at hour 0 + 3 x 24 = **72**.

In [26]:
# 1. One row per received offer, with its expiry time
received = (
    events.loc[events['event'].eq('offer received'), ['customer_id', 'offer_id', 'time']]  # keep only "offer received" rows
    .rename(columns={'time': 'received_time'})    # clear name, so it is not confused with view or completion time
    .merge(offers[['offer_id', 'offer_type', 'difficulty', 'reward', 'duration']],
           on='offer_id', how='left')             # add offer details from the offers table
)

# deadline in hours: duration is in days, time is in hours
received['window_end'] = received['received_time'] + received['duration'] * 24

# merge_asof (next step) needs the table sorted by time; the id gives each receipt a unique label
received = received.sort_values('received_time').reset_index(drop=True)
received['received_event_id'] = received.index

### Step 2.2: attach each view and completion to one receipt
The events table holds views and completions as separate rows. We need to decide **which received offer each one belongs to**.

`pd.merge_asof` is a "nearest match" merge. Example: customer D receives offer Y at hour 0 and again at hour 100, and views it at hour 120. Both receipts came before hour 120, and `merge_asof` picks the latest one (hour 100). The view is counted once, not twice.

An event is then kept only if it happened **before the offer expired**. Example: a view at hour 100 for an offer that expired at hour 72 is dropped.

In [27]:
# 2. Attach each view and each completion to exactly ONE received offer
def attach_to_receipt(event_name):
    ev = (events.loc[events['event'].eq(event_name), ['customer_id', 'offer_id', 'time']]
          .sort_values('time'))                  # merge_asof needs both tables sorted by time

    # merge_asof = "nearest match" merge. For each event, find the most recent receipt
    # (same customer, same offer) at or before the event time (direction='backward').
    m = pd.merge_asof(
        ev,
        received[['received_event_id', 'customer_id', 'offer_id', 'received_time', 'window_end']],
        left_on='time', right_on='received_time',
        by=['customer_id', 'offer_id'],
        direction='backward')

    # keep the event only if (a) a receipt was found and (b) the event happened before that offer expired
    in_window = m['received_event_id'].notna() & (m['time'] <= m['window_end'])

    n_attached = in_window.sum()
    print(f"{event_name}: {n_attached} of {len(m)} attached to a receipt; {len(m) - n_attached} outside any window")
    return m[in_window]

views = attach_to_receipt('offer viewed')
completions = attach_to_receipt('offer completed')

offer viewed: 48843 of 49860 attached to a receipt; 1017 outside any window
offer completed: 32070 of 32070 attached to a receipt; 0 outside any window


### Step 2.3: first view, and completions after a view
- `first_view`: one receipt can have several views. We keep the earliest, because we care about when the customer first knew about the offer.
- `first_any_completion`: the earliest completion, viewed or not.
- `influenced`: a completion counts only if it is at or after the first view. Example: viewed at hour 10 and completed at hour 30 (30 >= 10) is influenced. Completed at hour 20 with no view is not.

In [28]:
# 3. First view per receipt, and completions that follow a view

# one receipt can have several views: keep the earliest
first_view = views.groupby('received_event_id')['time'].min().rename('first_view_time')

# earliest completion per receipt, whether or not the offer was viewed
first_any_completion = (completions.groupby('received_event_id')['time'].min()
                        .rename('first_completion_any'))

# inner merge: completions on receipts that were never viewed drop out here
inf = completions.merge(first_view, on='received_event_id')

# influenced = completion at or after the first view (>= because time is in whole hours)
inf = inf[inf['time'] >= inf['first_view_time']]

first_influenced = (inf.groupby('received_event_id')['time'].min()
                    .rename('influenced_completion_time'))

### Step 2.4: build the funnel table and label each row
The flags (`viewed`, `completed_any`, `influenced_completion`) are True/False columns. `funnel_stage` combines them into one label. Each receipt gets exactly one stage:

| Stage | Meaning |
|---|---|
| received only | not viewed, not completed |
| viewed only | viewed, never completed |
| completed without view | completed, but not after a view (a coincidental completion, still rewarded) |
| completed after view | viewed, then completed (influenced) |

Because the stages are mutually exclusive, their counts add up to the number of receipts.

In [30]:
# 4. Build the final funnel table and label each row

# start from the receipts table and add the three time columns (left join: receipts with no match get NaN)
funnel = received.set_index('received_event_id').join([first_view, first_any_completion, first_influenced])

# notna() turns "does a time exist?" into True/False
funnel['viewed'] = funnel['first_view_time'].notna()
funnel['completed_any'] = funnel['first_completion_any'].notna()
funnel['influenced_completion'] = funnel['influenced_completion_time'].notna()

# np.select works like if / elif / else: the FIRST true condition wins, so the order matters
funnel['funnel_stage'] = np.select(
    [funnel['influenced_completion'],                                   # viewed, then completed
     funnel['completed_any'] & ~funnel['influenced_completion'],        # completed, but no view before it
     funnel['viewed']],                                                 # viewed, never completed
    ['completed after view', 'completed without view', 'viewed only'],
    default='received only')

In [31]:
# quick look at the funnel table
print(funnel)

                                        customer_id  \
received_event_id                                     
0                  78afa995795e4d85b5d9ceeca43f5fef   
1                  e2127556f4f64592b11af22de27a7932   
2                  389bc3fa690240e798340f5a15918d5c   
3                  2eeac8d8feae4a8cad5a6af0499a211d   
4                  aa4862eba776480b8bb9c68455b8c2e1   
...                                             ...   
66496              d087c473b4d247ccb0abfef59ba12b0e   
66497              cb23b66c56f64b109d673d5e56574529   
66498              6d5f3a774f3d4714ab0c092238f3a1d7   
66499              9dc1421481194dcd9400aec7c9ae6366   
66500              e4052622e5ba45a8b96b59aba68cf068   

                                           offer_id  received_time  \
received_event_id                                                    
0                  9b98b8c7a33c4b65b9aebfe6a799e6d9              0   
1                  2906b810c7d4411798c6938adc9daaa5              0   
2   

In [32]:
# 5. Sanity check: the funnel must have exactly one row per 'offer received' event
# (a bad merge could duplicate rows and silently inflate every count)
assert len(funnel) == events['event'].eq('offer received').sum()

### Step 2.5: summary by offer type
Rates use all received offers as the denominator:
- `view_rate` = viewed / received
- `influenced_rate` = influenced completions / received

**Note:** Informational offers have no completion event, so their 0% influenced rate does not mean the offers failed. There is nothing to measure. They are analysed separately in `04_informational_offer`.

In [21]:
# 6. Summary table by offer type
# summing a True/False column counts the True values
summary = funnel.groupby('offer_type')[['viewed', 'completed_any', 'influenced_completion']].sum()
summary.insert(0, 'received', funnel.groupby('offer_type').size())

summary['view_rate'] = summary['viewed'] / summary['received']
summary['influenced_rate'] = summary['influenced_completion'] / summary['received']

# note: informational offers show 0 completions because they have no 'offer completed' events (see 04)
summary.round(3)

,received,viewed,completed_any,influenced_completion,view_rate,influenced_rate
offer_type,,,,,,
bogo,26537,21865,15100,10647,0.824,0.401
discount,26664,18393,16900,11702,0.690,0.439
informational,13300,8585,0,0,0.645,0.000


## 3. Save the funnel
The funnel is saved so later notebooks can load it without repeating this logic.

In [33]:
# 7. Save the funnel
# received_event_id is currently the index; make it a normal column so it survives the CSV
funnel_out = funnel.reset_index()
funnel_out.to_csv('../data/cleaned/offer_funnel.csv', index=False)
print("offer_funnel:", funnel_out.shape)

offer_funnel: (66501, 16)


In [34]:
# check the saved file reloads with the same number of rows
check = pd.read_csv('../data/cleaned/offer_funnel.csv')
assert len(check) == len(funnel)

In [35]:
# True/False columns should come back as bool; time columns as numbers (float, because some are empty)
print(check.dtypes)

received_event_id               int64
customer_id                    object
offer_id                       object
received_time                   int64
offer_type                     object
difficulty                      int64
reward                          int64
duration                        int64
window_end                      int64
first_view_time               float64
first_completion_any          float64
influenced_completion_time    float64
viewed                           bool
completed_any                    bool
influenced_completion            bool
funnel_stage                   object
dtype: object


## 4. Findings, assumptions and limitations

### Findings (what the data shows)
- **BOGO offers are viewed more often than discount offers:** 82.4% vs 69.0% of received offers. Informational offers: 64.5%.
- **Influenced completion rate is similar for BOGO and discount** when measured against all received offers (40.1% vs 43.9%). It differs more once an offer is viewed: 48.7% of viewed BOGO offers and 63.6% of viewed discount offers were completed afterwards.
- **About 30% of completed offers had no view before the completion** (9,651 of 32,000 completed receipts). The customer met the spending threshold and earned the reward, but had not seen the offer first.
- **The two BOGO and discount offer types complete at very different stages.** For BOGO, 4,453 of 15,100 completions had no prior view (29.5%). For discount, 5,198 of 16,900 (30.8%).
- **Informational offers show no completions** because the dataset has no completion event for them. This is a data limitation, not a result.

### Interpretation (reasonable, but not proven)
- The higher BOGO view rate may reflect the channels used to send the offers. This was checked for the two informational offers (the one sent through social was viewed far more often) but not yet for BOGO and discount.
- The higher view-to-completion rate for discount offers may reflect differences in difficulty and reward between the offers, not a better offer type. Offers were not compared at equal difficulty.
- The ~30% of completions without a prior view suggests some rewards go to customers who would have bought anyway. That would mean reward spend is partly not driven by the offer.

### Assumptions
- **Window:** an offer is active from the time it is received until `received_time + duration x 24` hours.
- **Overlapping receipts:** if a customer received the same offer more than once, each view or completion belongs to the **most recent** receipt before it.
- **Same-hour ties:** a view and a completion in the same hour count as viewed-then-completed (`>=`).
- **First view only:** only the earliest view in the window is used.
- **Excluded data:** customers with incomplete profiles (12.8%) and duplicate events (374 rows) were removed in `01_data_cleaning`.

### Limitations
- **Sequence is not cause.** "Influenced" only means the customer viewed the offer before completing it. There is no control group, so we cannot tell what they would have done without the offer.
- **Same-hour ordering is unknown.** Times are in whole hours, so a view and completion in the same hour could have happened in either order (2,695 completions fall in this case in the strict version of the logic).
- **Latest-receipt rule can misplace an event.** If a customer viewed the first of two overlapping receipts and completed after the second arrived, the completion is matched to the second receipt, which had no view. It is then counted as "completed without view".
- **Some offers run past the end of the data.** The data ends at hour 714, and 13.4% of receipts (8,940) expire after that. Their completions after hour 714 are not observed. The completion rate for BOGO and discount is similar for these offers (59.3% vs 60.3%), so any bias looks small, but it is not ruled out.
- **Views after expiry are ignored.** 1,017 of the 49,860 views (2.0%) happened after the offer expired and are not counted.
- **Informational offers cannot be measured here.** They are analysed separately in `04_informational_offer`.
- **A 30-day snapshot.** Results describe this period only, and customers with complete profiles only.